# Visualización de datos — Ejemplos

**Módulo 2 — Transformación y visualización de datos · Curso Analítica de Datos**

Este notebook acompaña las diapositivas [`2.3_Visualizacion_datos.pdf`](2.3_Visualizacion_datos.pdf) y lleva a código lo que allí se explica, usando un dataset real muy popular entre los videojugadores: el **Video Game Sales Dataset** de Kaggle.

## Contenido

1. **Cargar el dataset de Kaggle**: ventas de más de 16.000 videojuegos por región.
2. **Visualización univariada**: histogramas.
3. **Forma de la distribución**: asimetría, media vs. mediana.
4. **Diagrama de caja y detección de valores atípicos** (regla del IQR).
5. **Variables categóricas**: conteos, proporciones y barras apiladas.
6. **Relaciones bivariantes**: gráfico de dispersión.
7. **Series temporales y mapa de calor de correlaciones**.
8. **Actividad para discutir en clase**.

> 💡 La sección 1 **requiere una cuenta de Kaggle y una clave de API** (gratis) — el proceso está explicado paso a paso en el notebook `1.6` del Módulo 1 y se resume aquí también.

In [ ]:
# Librerías que usaremos en todo el notebook
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

pd.set_option('display.max_columns', 20)
plt.rcParams['figure.figsize'] = (8, 5)
sns.set_style('whitegrid')

print('Librerías cargadas correctamente ✅')
print('pandas', pd.__version__, '· seaborn', sns.__version__)

---
## 1. Cargar el dataset de Kaggle: *Video Game Sales*

El [**Video Game Sales Dataset**](https://www.kaggle.com/datasets/gregorut/videogamesales) reúne más de 16.000 videojuegos con sus ventas (en millones de copias) en Norteamérica, Europa, Japón, el resto del mundo y a nivel global, además de su plataforma, género, año de lanzamiento y editora. Es un dataset ideal para visualización: tiene variables numéricas muy asimétricas (unos pocos juegos venden muchísimo más que el resto), variables categóricas con muchas categorías (género, plataforma) y una dimensión temporal (año de lanzamiento).

### Cómo obtener tu clave de API de Kaggle (si no la tienes de un notebook anterior)

1. Crea una cuenta gratuita en [kaggle.com](https://www.kaggle.com/) si no tienes una.
2. Ve a tu perfil → **Settings** → sección **API** → botón **"Create New Token"**. Esto descarga un archivo `kaggle.json` con tus credenciales.
3. Instala la librería oficial de Kaggle (si no la tienes): `pip install kagglehub`
4. La primera vez que ejecutes la celda de descarga, `kagglehub` te pedirá autenticarte (puede abrir el navegador, o puedes colocar el archivo `kaggle.json` descargado en `~/.kaggle/kaggle.json`).

In [ ]:
import kagglehub
import os

try:
    ruta_dataset = kagglehub.dataset_download('gregorut/videogamesales')
    print('Dataset descargado en:', ruta_dataset)
except Exception as error:
    print('❌ No se pudo descargar el dataset de Kaggle.')
    print('Verifica tu conexión a internet y que tu API key esté configurada (ver celda anterior).')
    print('Detalle del error:', error)
    raise

archivos_csv = [f for f in os.listdir(ruta_dataset) if f.endswith('.csv')]
print('Archivos CSV encontrados:', archivos_csv)

ruta_csv = os.path.join(ruta_dataset, archivos_csv[0])
juegos = pd.read_csv(ruta_csv)

# Un puñado de filas no traen año de lanzamiento; las descartamos para las
# secciones que dependen del tiempo, pero conservamos el resto del dataset.
juegos = juegos.dropna(subset=['Year', 'Genre'])
juegos['Year'] = juegos['Year'].astype(int)

print(f'\nDataset cargado: {juegos.shape[0]} filas x {juegos.shape[1]} columnas')
juegos.head()

In [ ]:
juegos.info()

---
## 2. Visualización univariada: histogramas

Como en la diapositiva 2, exploramos la distribución de `Global_Sales` (ventas globales, en millones de copias) probando **distintos números de intervalos (bins)** para ver cómo cambia lo que percibimos de la distribución.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].hist(juegos['Global_Sales'], bins=10, color='#1565c0', edgecolor='white')
axes[0].set_title('Con 10 intervalos (bins)')
axes[0].set_xlabel('Ventas globales (millones de copias)')
axes[0].set_ylabel('Cantidad de juegos')

axes[1].hist(juegos['Global_Sales'], bins=50, color='#1565c0', edgecolor='white')
axes[1].set_title('Con 50 intervalos (bins)')
axes[1].set_xlabel('Ventas globales (millones de copias)')

plt.tight_layout()
plt.show()

print('Con pocos bins, casi todo se ve amontonado en la primera barra.')
print('Con más bins vemos mejor que la inmensa mayoría de juegos vende muy poco,')
print('y una minoría absoluta vende cifras enormes: una "cola" muy larga hacia la derecha.')

---
## 3. Forma de la distribución: asimetría

Siguiendo la diapositiva 3, comparamos la **media** y la **mediana** de `Global_Sales`, y calculamos su coeficiente de asimetría.

In [ ]:
media = juegos['Global_Sales'].mean()
mediana = juegos['Global_Sales'].median()
asimetria = juegos['Global_Sales'].skew()

print(f'Media:     {media:.3f} millones de copias')
print(f'Mediana:   {mediana:.3f} millones de copias')
print(f'Asimetría: {asimetria:.2f}  (un valor tan alto y positivo confirma una cola muy larga hacia la derecha)')

print('\nMedia > mediana ⇒ asimetría positiva, tal como en el ejemplo de salarios/precios de vivienda de la diapositiva 3.')
print('\nLos 5 juegos más vendidos de la historia (los "outliers" que arrastran la media hacia arriba):')
juegos.nlargest(5, 'Global_Sales')[['Name', 'Platform', 'Year', 'Global_Sales']]

---
## 4. Diagrama de caja y detección de valores atípicos

Aplicamos la **regla del IQR** de la diapositiva 4-5 para detectar valores atípicos en `Global_Sales`.

In [ ]:
q1, q3 = juegos['Global_Sales'].quantile([0.25, 0.75])
iqr = q3 - q1
limite_superior = q3 + 1.5 * iqr

atipicos = juegos[juegos['Global_Sales'] > limite_superior]

print(f'Q1 = {q1:.3f}, Q3 = {q3:.3f}, IQR = {iqr:.3f}')
print(f'Límite superior (Q3 + 1.5×IQR) = {limite_superior:.3f} millones de copias')
print(f'\nCon esta regla, {len(atipicos)} de {len(juegos)} juegos ({len(atipicos) / len(juegos):.1%}) son "atípicos".')

fig, ax = plt.subplots()
ax.boxplot(juegos['Global_Sales'], vert=False)
ax.set_xlabel('Ventas globales (millones de copias)')
ax.set_title('Diagrama de caja — Global_Sales')
plt.tight_layout()
plt.show()

⚠️ **Punto de discusión** (como en la diapositiva 5, "¿error o dato válido?"): la regla del IQR marca más del 10% de los juegos como "atípicos". Aquí ninguno es un error de captura — es simplemente que el mercado de videojuegos está dominado por unos pocos éxitos masivos (*Wii Sports*, *Super Mario Bros.*...) frente a una enorme mayoría de juegos con ventas modestas. Esto ilustra bien la advertencia de la diapositiva: **la regla IQR es un punto de partida, no una verdad absoluta — siempre hay que investigar qué representa cada valor antes de descartarlo.**

---
## 5. Variables categóricas: conteos y proporciones

Como en la diapositiva 7: contamos juegos por género, y comparamos **proporciones** entre las plataformas más populares con una tabla de contingencia y barras apiladas al 100%.

In [ ]:
conteo_genero = juegos['Genre'].value_counts()

fig, ax = plt.subplots()
ax.bar(conteo_genero.index, conteo_genero.values, color='#2e7d32')
ax.set_title('Cantidad de juegos por género')
ax.set_ylabel('Cantidad de juegos')
ax.tick_params(axis='x', rotation=40)
plt.tight_layout()
plt.show()

print(f"Género más común: '{conteo_genero.idxmax()}' con {conteo_genero.max()} juegos.")

In [ ]:
# Tabla de contingencia: proporción (%) de cada género dentro de las 5 plataformas más frecuentes
top_plataformas = juegos['Platform'].value_counts().head(5).index
subset = juegos[juegos['Platform'].isin(top_plataformas)]

tabla_contingencia = pd.crosstab(subset['Platform'], subset['Genre'], normalize='index') * 100
print('Tabla de contingencia (% de juegos de cada género, por plataforma):')
tabla_contingencia.round(1)

In [ ]:
fig, ax = plt.subplots(figsize=(9, 5))
tabla_contingencia.plot(kind='bar', stacked=True, ax=ax, colormap='tab20')
ax.set_ylabel('% de juegos')
ax.set_title('Distribución de géneros por plataforma (barras apiladas al 100%)')
ax.legend(bbox_to_anchor=(1.02, 1), loc='upper left', fontsize=8)
plt.tight_layout()
plt.show()

print('Las barras apiladas nos dejan comparar la MEZCLA de géneros entre plataformas')
print('de tamaños muy distintos (no solo cuántos juegos tiene cada una).')

---
## 6. Relaciones bivariantes: gráfico de dispersión

Siguiendo la diapositiva 8, exploramos la relación entre las ventas en Norteamérica y en Europa.

In [ ]:
fig, ax = plt.subplots()
ax.scatter(juegos['NA_Sales'], juegos['EU_Sales'], alpha=0.3, s=12, color='#6a1b9a')
ax.set_xlabel('Ventas en Norteamérica (millones)')
ax.set_ylabel('Ventas en Europa (millones)')
ax.set_title('Ventas en Norteamérica vs. Europa, por juego')
plt.tight_layout()
plt.show()

correlacion = juegos['NA_Sales'].corr(juegos['EU_Sales'])
print(f'Correlación: {correlacion:.3f}')
print('\nDirección: positiva (los juegos que venden mucho en Norteamérica también tienden a vender mucho en Europa).')
print('Forma: aproximadamente lineal, con más dispersión a medida que crecen las ventas.')
print('Recordemos la advertencia de la diapositiva 8: esta asociación NO demuestra causalidad')
print('— ambas ventas probablemente dependen de una tercera variable en común: qué tan bueno/popular es el juego.')

---
## 7. Series temporales y mapa de calor de correlaciones

Como en la diapositiva 9: una serie temporal debe **conservar el orden cronológico**, y un mapa de calor resume muchas correlaciones a la vez.

In [ ]:
# Nos quedamos con 1980-2016: el dataset tiene muy pocos juegos registrados
# después de 2016, lo que distorsionaría la serie si los incluyéramos.
juegos_periodo = juegos[(juegos['Year'] >= 1980) & (juegos['Year'] <= 2016)]
ventas_por_anio = juegos_periodo.groupby('Year')['Global_Sales'].sum().sort_index()

fig, ax = plt.subplots()
ax.plot(ventas_por_anio.index, ventas_por_anio.values, marker='o', color='#c62828')
ax.set_xlabel('Año de lanzamiento')
ax.set_ylabel('Ventas globales totales (millones de copias)')
ax.set_title('Ventas globales de videojuegos por año (1980–2016)')
plt.tight_layout()
plt.show()

print(f'Año con más ventas globales: {ventas_por_anio.idxmax()} ({ventas_por_anio.max():.0f} millones de copias)')

In [ ]:
columnas_ventas = ['NA_Sales', 'EU_Sales', 'JP_Sales', 'Other_Sales', 'Global_Sales']
correlaciones = juegos[columnas_ventas].corr()

fig, ax = plt.subplots()
sns.heatmap(correlaciones, annot=True, fmt='.2f', cmap='Greens', vmin=-1, vmax=1, ax=ax)
ax.set_title('Mapa de calor de correlaciones entre ventas por región')
plt.tight_layout()
plt.show()

print('Japón (JP_Sales) es la región que menos se correlaciona con las demás:')
print('el mercado japonés de videojuegos tiene gustos bastante distintos al resto del mundo.')

---
## 8. Para pensar y discutir en clase

Siguiendo la estructura de la actividad integradora de las diapositivas (**Pregunta → Evidencia → Conclusión**):

1. **Pregunta**: ¿Qué género de videojuego domina cada región (Norteamérica, Europa, Japón)? ¿Son los mismos géneros en las tres regiones?
2. **Evidencia**: construye una visualización univariada (por ejemplo, ventas totales por género) y una bivariada (por ejemplo, un gráfico de barras agrupadas comparando `NA_Sales`, `EU_Sales` y `JP_Sales` por género) que respondan la pregunta.
3. **Conclusión**: argumenta con tus gráficos. ¿Hay algún género que sea mucho más popular en una región que en las demás? ¿Se te ocurre una explicación cultural o de mercado para esa diferencia? Recuerda distinguir correlación de causalidad.

No hay una única respuesta "correcta": lo importante es elegir el gráfico adecuado y leer sus ejes y unidades correctamente, tal como piden los criterios de revisión de la diapositiva.

---
## Cierre

En este notebook llevamos a código las ideas de las diapositivas [`2.3_Visualizacion_datos.pdf`](2.3_Visualizacion_datos.pdf), usando el **Video Game Sales Dataset**:

- Cómo el número de **bins** de un histograma cambia lo que percibimos de una distribución.
- Cómo identificar **asimetría** comparando media y mediana.
- Cómo aplicar la **regla del IQR** para detectar valores atípicos, y por qué hay que investigarlos antes de descartarlos.
- Cómo resumir variables categóricas con **conteos**, **tablas de contingencia** y **barras apiladas al 100%**.
- Cómo leer un **gráfico de dispersión** (dirección, forma, atípicos) sin confundir correlación con causalidad.
- Cómo construir una **serie temporal** respetando el orden cronológico y un **mapa de calor de correlaciones**.

### Recursos adicionales
- [Video Game Sales Dataset en Kaggle](https://www.kaggle.com/datasets/gregorut/videogamesales)
- [Documentación de `pandas.crosstab`](https://pandas.pydata.org/docs/reference/api/pandas.crosstab.html)
- [Documentación de `seaborn.heatmap`](https://seaborn.pydata.org/generated/seaborn.heatmap.html)